# EMT 3-phase Generic Two-Terminal V-Type SSN

This notebook compares a reference EMT model built from standard MNA components against two SSN-based variants:

1. **SSN load**: the RLC load is replaced by a generic two-terminal V-type SSN component
2. **SSN branch**: the series RL branch is replaced by a generic two-terminal V-type SSN component

The SSN cases are validated against the reference simulation using logged CSV results.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import dpsimpy

emt = dpsimpy.emt
ph3 = emt.ph3

Simulation = dpsimpy.Simulation
SystemTopology = dpsimpy.SystemTopology
Logger = dpsimpy.Logger
Domain = dpsimpy.Domain
Solver = dpsimpy.Solver
PhaseType = dpsimpy.PhaseType

## Parameters

In [ ]:
time_step = 1e-4
final_time = 0.2
frequency = 50.0

source_voltage = 1.0 * np.exp(1j * 0.0)

i3 = np.eye(3)

branch_resistance = 0.2 * i3
branch_inductance = 0.02 * i3

load_resistance = 1.0 * i3
load_inductance = 0.05 * i3
load_capacitance = 100e-6 * i3

## Helpers

In [ ]:
def abc(phasor):
    return np.array(
        [
            [phasor],
            [phasor * np.exp(-1j * 2 * np.pi / 3)],
            [phasor * np.exp(+1j * 2 * np.pi / 3)],
        ],
        dtype=np.complex128,
    )


def create_rlc_load_ssn_matrices():
    inv_l = np.linalg.inv(load_inductance)
    inv_c = np.linalg.inv(load_capacitance)

    # x = [uC_abc; i_abc], u = v_abc, y = i_abc
    a = np.zeros((6, 6))
    a[0:3, 3:6] = inv_c
    a[3:6, 0:3] = -inv_l
    a[3:6, 3:6] = -inv_l @ load_resistance

    b = np.zeros((6, 3))
    b[3:6, 0:3] = inv_l

    c = np.zeros((3, 6))
    c[0:3, 3:6] = i3

    d = np.zeros((3, 3))
    return a, b, c, d


def create_rl_branch_ssn_matrices():
    inv_l = np.linalg.inv(branch_inductance)

    # x = i_abc, u = v_abc, y = i_abc
    a = -inv_l @ branch_resistance
    b = inv_l
    c = i3.copy()
    d = np.zeros((3, 3))
    return a, b, c, d

## Reference case: standard MNA components

In [ ]:
def run_mna_case():
    sim_name = "EMT_Ph3_MNAComponents"

    n1 = emt.SimNode("n1", PhaseType.ABC)
    n2 = emt.SimNode("n2", PhaseType.ABC)
    n3 = emt.SimNode("n3", PhaseType.ABC)
    n4 = emt.SimNode("n4", PhaseType.ABC)
    n5 = emt.SimNode("n5", PhaseType.ABC)

    vs = ph3.VoltageSource("VS")
    vs.set_parameters(abc(source_voltage), frequency)

    r_branch = ph3.Resistor("RBranch")
    r_branch.set_parameters(branch_resistance)

    l_branch = ph3.Inductor("LBranch")
    l_branch.set_parameters(branch_inductance)

    r_load = ph3.Resistor("RLoad")
    r_load.set_parameters(load_resistance)

    l_load = ph3.Inductor("LLoad")
    l_load.set_parameters(load_inductance)

    c_load = ph3.Capacitor("CLoad")
    c_load.set_parameters(load_capacitance)

    vs.connect([emt.SimNode.gnd, n1])
    r_branch.connect([n1, n2])
    l_branch.connect([n2, n3])

    r_load.connect([n3, n4])
    l_load.connect([n4, n5])
    c_load.connect([n5, emt.SimNode.gnd])

    system = SystemTopology(
        frequency,
        [n1, n2, n3, n4, n5],
        [vs, r_branch, l_branch, r_load, l_load, c_load],
    )

    Logger.set_log_dir("logs/" + sim_name)
    logger = Logger(sim_name)
    logger.log_attribute("v3", n3.attr("v"))
    logger.log_attribute("i_branch", l_branch.attr("i_intf"))
    logger.log_attribute("i_load", r_load.attr("i_intf"))

    sim = Simulation(sim_name)
    sim.set_system(system)
    sim.add_logger(logger)
    sim.set_domain(Domain.EMT)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.run()

## SSN load case

In [ ]:
def run_rlc_load_ssn_case():
    sim_name = "EMT_Ph3_RLCLoadSSN"

    n1 = emt.SimNode("n1", PhaseType.ABC)
    n2 = emt.SimNode("n2", PhaseType.ABC)
    n3 = emt.SimNode("n3", PhaseType.ABC)
    # n4, n5 do not exist because of SSN

    a, b, c, d = create_rlc_load_ssn_matrices()

    vs = ph3.VoltageSource("VS")
    vs.set_parameters(abc(source_voltage), frequency)

    r_branch = ph3.Resistor("RBranch")
    r_branch.set_parameters(branch_resistance)

    l_branch = ph3.Inductor("LBranch")
    l_branch.set_parameters(branch_inductance)

    rlc_load = ph3.GenericTwoTerminalVTypeSSN("RLCLoad")
    rlc_load.set_parameters(a, b, c, d)

    vs.connect([emt.SimNode.gnd, n1])
    r_branch.connect([n1, n2])
    l_branch.connect([n2, n3])
    rlc_load.connect([n3, emt.SimNode.gnd])

    system = SystemTopology(
        frequency,
        [n1, n2, n3],
        [vs, r_branch, l_branch, rlc_load],
    )

    Logger.set_log_dir("logs/" + sim_name)
    logger = Logger(sim_name)
    logger.log_attribute("v3", n3.attr("v"))
    logger.log_attribute("i_branch", l_branch.attr("i_intf"))
    logger.log_attribute("i_load", rlc_load.attr("i_intf"))

    sim = Simulation(sim_name)
    sim.set_system(system)
    sim.add_logger(logger)
    sim.set_domain(Domain.EMT)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.run()

## SSN branch case

In [ ]:
def run_rl_branch_ssn_case():
    sim_name = "EMT_Ph3_RLBranchSSN"

    n1 = emt.SimNode("n1", PhaseType.ABC)
    # n2 does not exist because of SSN
    n3 = emt.SimNode("n3", PhaseType.ABC)
    n4 = emt.SimNode("n4", PhaseType.ABC)
    n5 = emt.SimNode("n5", PhaseType.ABC)

    a, b, c, d = create_rl_branch_ssn_matrices()

    vs = ph3.VoltageSource("VS")
    vs.set_parameters(abc(source_voltage), frequency)

    rl_branch = ph3.GenericTwoTerminalVTypeSSN("RLBranch")
    rl_branch.set_parameters(a, b, c, d)

    r_load = ph3.Resistor("RLoad")
    r_load.set_parameters(load_resistance)

    l_load = ph3.Inductor("LLoad")
    l_load.set_parameters(load_inductance)

    c_load = ph3.Capacitor("CLoad")
    c_load.set_parameters(load_capacitance)

    vs.connect([emt.SimNode.gnd, n1])
    rl_branch.connect([n1, n3])

    r_load.connect([n3, n4])
    l_load.connect([n4, n5])
    c_load.connect([n5, emt.SimNode.gnd])

    system = SystemTopology(
        frequency,
        [n1, n3, n4, n5],
        [vs, rl_branch, r_load, l_load, c_load],
    )

    Logger.set_log_dir("logs/" + sim_name)
    logger = Logger(sim_name)
    logger.log_attribute("v3", n3.attr("v"))
    logger.log_attribute("i_branch", rl_branch.attr("i_intf"))
    logger.log_attribute("i_load", r_load.attr("i_intf"))

    sim = Simulation(sim_name)
    sim.set_system(system)
    sim.add_logger(logger)
    sim.set_domain(Domain.EMT)
    sim.set_time_step(time_step)
    sim.set_final_time(final_time)
    sim.run()

## Run simulations

In [ ]:
run_mna_case()
run_rlc_load_ssn_case()
run_rl_branch_ssn_case()

## Load CSV logs

In [ ]:
ref = pd.read_csv(
    Path("logs/EMT_Ph3_MNAComponents/EMT_Ph3_MNAComponents.csv"), skipinitialspace=True
)
rlc_ssn = pd.read_csv(
    Path("logs/EMT_Ph3_RLCLoadSSN/EMT_Ph3_RLCLoadSSN.csv"), skipinitialspace=True
)
rl_branch_ssn = pd.read_csv(
    Path("logs/EMT_Ph3_RLBranchSSN/EMT_Ph3_RLBranchSSN.csv"), skipinitialspace=True
)

## Assertions

In [ ]:
def sig(df, name):
    cols = [c for c in df.columns if c == name or c.startswith(name + "_")]
    return df[cols].to_numpy()


t_ref = ref.iloc[:, 0].to_numpy()
t_rlc = rlc_ssn.iloc[:, 0].to_numpy()
t_rlb = rl_branch_ssn.iloc[:, 0].to_numpy()

np.testing.assert_allclose(t_ref, t_rlc)
np.testing.assert_allclose(t_ref, t_rlb)

atol = 1e-9
rtol = 1e-6

np.testing.assert_allclose(sig(ref, "v3"), sig(rlc_ssn, "v3"), atol=atol, rtol=rtol)
np.testing.assert_allclose(
    sig(ref, "i_branch"), sig(rlc_ssn, "i_branch"), atol=atol, rtol=rtol
)
np.testing.assert_allclose(
    sig(ref, "i_load"), sig(rlc_ssn, "i_load"), atol=atol, rtol=rtol
)

np.testing.assert_allclose(
    sig(ref, "v3"), sig(rl_branch_ssn, "v3"), atol=atol, rtol=rtol
)
np.testing.assert_allclose(
    sig(ref, "i_branch"), sig(rl_branch_ssn, "i_branch"), atol=atol, rtol=rtol
)
np.testing.assert_allclose(
    sig(ref, "i_load"), sig(rl_branch_ssn, "i_load"), atol=atol, rtol=rtol
)

print("All assertions passed.")

## Plot comparison

In [ ]:
phase = 0

plt.figure(figsize=(10, 5))

plt.plot(t_ref, sig(ref, "i_branch")[:, phase], label="ref i_branch")
plt.plot(t_rlc, sig(rlc_ssn, "i_load")[:, phase], "--", label="RLCLoadSSN i_load")
plt.plot(
    t_rlb, sig(rl_branch_ssn, "i_branch")[:, phase], ":", label="RLBranchSSN i_branch"
)

plt.xlabel("time [s]")
plt.ylabel("current")
plt.title("Currents, phase A")
plt.grid(True)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))

plt.plot(t_ref, sig(ref, "v3")[:, phase], label="ref v3")
plt.plot(t_rlc, sig(rlc_ssn, "v3")[:, phase], "--", label="RLCLoadSSN v3")
plt.plot(t_rlb, sig(rl_branch_ssn, "v3")[:, phase], ":", label="RLBranchSSN v3")

plt.xlabel("time [s]")
plt.ylabel("voltage")
plt.title("Voltages, phase A")
plt.grid(True)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()